In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../outputs/control/20261011_180526/results.csv")

In [9]:
df[['model','train_time_mean_s',
       'inference_time_mean_s',
       'fit_create_cluster_datasets_mean_s', 'fit_find_best_k_mean_s',
    'fit_fuzzy_clustering_mean_s',
        'fit_total_fit_mean_s',
        'fit_train_classifiers_mean_s',
        'predict_aggregation_mean_s',
        'predict_class_remapping_mean_s',
        'predict_cluster_weights_mean_s',
       'predict_total_predict_proba_mean_s',

       'predict_tree_predict_proba_mean_s',]].iloc[[0]]

,model,train_time_mean_s,inference_time_mean_s,fit_create_cluster_datasets_mean_s,fit_find_best_k_mean_s,fit_fuzzy_clustering_mean_s,fit_total_fit_mean_s,fit_train_classifiers_mean_s,predict_aggregation_mean_s,predict_class_remapping_mean_s,predict_cluster_weights_mean_s,predict_total_predict_proba_mean_s,predict_tree_predict_proba_mean_s
0,FuzzyEspecialistEnsembleClassifier,1.946277,0.019721,0.474662,0.33821,0.003933,1.946235,1.129198,0.000307,0.001613,0.000091,0.01968,0.016925


In [8]:
df[['model','accuracy_mean','accuracy_std', 'auc_mean','auc_std']].iloc[[0,1]]

,model,accuracy_mean,accuracy_std,auc_mean,auc_std
0,FuzzyEspecialistEnsembleClassifier,0.898000,0.006182,0.955979,0.006628
1,FastKSelectionEnsembleClassifier,0.901333,0.010873,0.956126,0.006836


In [4]:
df[['model', 'train_time_mean_s', 'train_time_std_s', 'inference_time_mean_s', 'inference_time_std_s', 'fit_find_best_k_mean_s', 'fit_find_best_k_std_s']]

,model,train_time_mean_s,train_time_std_s,inference_time_mean_s,inference_time_std_s,fit_find_best_k_mean_s,fit_find_best_k_std_s
0,FuzzyEspecialistEnsembleClassifier,1.946277,0.059948,0.019721,0.001229,0.338210,0.009573
1,FastKSelectionEnsembleClassifier,1.642488,0.042942,0.019261,0.000706,0.052549,0.002080
2,CVKSelectionEnsembleClassifier,8.300425,0.377089,0.020035,0.001140,6.669403,0.386843
3,SupervisedEntropyKSelectionEnsembleClassifier,1.649404,0.047337,0.020131,0.002110,0.044257,0.004281
4,StabilityKSelectionEnsembleClassifier,1.751331,0.092457,0.017940,0.000783,1.069171,0.035041
5,GapStatisticKSelectionEnsembleClassifier,2.110675,0.056229,0.019423,0.001127,0.546729,0.029749


In [7]:
from itertools import combinations
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind_from_stats

def pairwise_tests(df, metrics=("accuracy", "auc", "train_time", "inference_time",
                                "fit_find_best_k", "fit_train_classifiers"), alpha=0.05):
    rows = []
    for metric in metrics:
        # column naming: accuracy_mean / train_time_mean_s / fit_find_best_k_mean_s ...
        mean_col = next(c for c in df.columns if c.startswith(metric + "_mean"))
        std_col  = mean_col.replace("_mean", "_std")

        for a, b in combinations(df.index, 2):
            ra, rb = df.loc[a], df.loc[b]
            na, nb = int(ra["n_folds_ok"]), int(rb["n_folds_ok"])
            if min(na, nb) < 2 or pd.isna(ra[mean_col]) or pd.isna(rb[mean_col]):
                continue
            # stored std is np.std (ddof=0) -> convert to sample std (ddof=1)
            sa = ra[std_col] * np.sqrt(na / (na - 1))
            sb = rb[std_col] * np.sqrt(nb / (nb - 1))
            t, p = ttest_ind_from_stats(ra[mean_col], sa, na,
                                        rb[mean_col], sb, nb, equal_var=False)
            rows.append({"metric": metric, "model_a": ra["model"], "model_b": rb["model"],
                         "mean_a": ra[mean_col], "mean_b": rb[mean_col],
                         "diff": ra[mean_col] - rb[mean_col], "t": t, "p": p})

    out = pd.DataFrame(rows)
    # Holm correction within each metric (many pairs -> many false positives otherwise)
    out["p_holm"] = out.groupby("metric")["p"].transform(
        lambda p: np.minimum(1, (p.rank(method="first").rsub(len(p) + 1) * p)
                             .reindex(p.sort_values().index).cummax().reindex(p.index)))
    out["significant"] = out["p_holm"] < alpha
    return out

df = pd.read_csv("../outputs/control/20261011_180526/results.csv")
print(pairwise_tests(df.iloc[[0,3]]).to_string(index=False))

               metric                            model_a                                       model_b   mean_a   mean_b      diff         t            p       p_holm  significant
             accuracy FuzzyEspecialistEnsembleClassifier SupervisedEntropyKSelectionEnsembleClassifier 0.898000 0.901333 -0.003333 -0.533002 6.121872e-01 6.121872e-01        False
                  auc FuzzyEspecialistEnsembleClassifier SupervisedEntropyKSelectionEnsembleClassifier 0.955979 0.956126 -0.000147 -0.030808 9.761780e-01 9.761780e-01        False
           train_time FuzzyEspecialistEnsembleClassifier SupervisedEntropyKSelectionEnsembleClassifier 1.946277 1.649404  0.296874  7.773212 7.128470e-05 7.128470e-05         True
       inference_time FuzzyEspecialistEnsembleClassifier SupervisedEntropyKSelectionEnsembleClassifier 0.019721 0.020131 -0.000409 -0.334960 7.483088e-01 7.483088e-01        False
      fit_find_best_k FuzzyEspecialistEnsembleClassifier SupervisedEntropyKSelectionEnsembleClassifi